# YOLOv5 Training on DOTA

this notebook goes through training yolov5 on the DOTA dataset and then using a selection algorithm based on pseudo-safety labels

## Train

In [ ]:
from ultralytics import YOLO

# Load a COCO-pretrained YOLOv5n model
model = YOLO("yolov5l.pt")

# train the model
model.train(
    data="DOTAv1.yaml", 
    epochs=100, 
    imgsz=1024,
    # batch=2,      
    # device=0,
    # workers=8
)

## Eval

In [ ]:
from ultralytics import YOLO

# load trained weights
model = YOLO("runs/detect/train4/weights/best.pt")

# Display model information (optional)
model.info()

YOLOv5l summary: 241 layers, 53,174,909 parameters, 0 gradients, 135.3 GFLOPs


(241, 53174909, 0, 135.345408)

In [3]:
# eval
val_metrics = model.val(data="DOTAv1.yaml", split="val", imgsz=1024)

print("VAL metrics:");  
print(f"mAP50-95: {val_metrics.box.map:.4f} | mAP50: {val_metrics.box.map50:.4f} | mAP75: {val_metrics.box.map75:.4f}")

Ultralytics 8.3.226  Python-3.9.18 torch-2.0.1 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
YOLOv5l summary (fused): 128 layers, 53,142,973 parameters, 0 gradients, 134.7 GFLOPs
val: Fast image access  (ping: 0.50.5 ms, read: 57.648.2 MB/s, size: 359.3 KB)
val: Scanning E:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\datasets\DOTAv1\labels\val.cache... 458 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 458/458  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 0.1it/s 3:270.5ss
                   all        458      28853      0.781      0.567      0.609      0.415
                 plane         70       2531       0.93      0.743      0.807      0.592
                  ship        108       8960      0.887      0.616      0.704      0.503
          storage tank         55       2888      0.935      0.345      0.536      0.336
      baseball diamond         53        214   

## Inference on ICG

In [3]:
from ultralytics import YOLO

# load your trained model
model = YOLO("runs/detect/train4/weights/best.pt")

# path to your test images (.jpg)
source = "../Data/training_set/slz_out/det_obb/splits_yolo_obb/images/test"

# safe class IDs according to DOTA YAML
SAFE_CLASS_IDS = [3, 4, 5, 6, 12, 13] # baseball diamond, tennis court, basketball court, ground track field, soccer ball field, roundabout

results = model.predict(
    source=source,
    imgsz=1024,
    conf=0.25,              # adjust if you want more/less boxes
    classes=SAFE_CLASS_IDS, # <– THIS filters to only the safe classes
    save=True,              # save annotated images
    save_txt=True,          # save YOLO-format labels
    save_conf=True,         # include confidence score at end of each line
    project="runs/detect_safe",
    name="yolov5l_dota",
)


image 1/40 e:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\compare\..\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\038.jpg: 704x1024 (no detections), 87.1ms
image 2/40 e:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\compare\..\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\044.jpg: 704x1024 (no detections), 27.6ms
image 3/40 e:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\compare\..\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\049.jpg: 704x1024 1 tennis court, 26.7ms
image 4/40 e:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\compare\..\Data\training_set\slz_out\det_obb\splits_yolo_obb\images\test\059.jpg: 704x1024 (no detections), 180.3ms
image 5/40 e:\Documents\updated uni work\KU PhD\Research\Projects\SafeLanding\UAV-Seg2Det-SafeLanding\compare\..\Data\training_set\

## Convert to OBB format 

In [4]:
from pathlib import Path

# --- UPDATE THESE ---
SRC_DIR = Path("runs/detect_safe/yolov5l_dota/labels")       # folder with AABB txts
DST_DIR = Path("../slz_obb/predictions/5l_dota")   # folder to write OBB txts
# --------------------

OVERRIDE_CLASSES = True

DST_DIR.mkdir(parents=True, exist_ok=True)

for txt_path in SRC_DIR.glob("*.txt"):
    lines_out = []
    for line in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s:
            continue
        parts = s.split()
        if len(parts) < 5:
            # not cls cx cy w h -> skip or warn
            print(f"[warn] unexpected format in {txt_path.name}: {s}")
            continue

        cls, cx, cy, w, h = parts[:5]
        if OVERRIDE_CLASSES:
            cls = "0"

        cx = float(cx); cy = float(cy)
        w  = float(w);  h  = float(h)

        # axis-aligned box corners in normalized coords
        x1 = cx - w / 2.0
        x2 = cx + w / 2.0
        y1 = cy - h / 2.0
        y2 = cy + h / 2.0

        # clamp to [0,1] just in case
        x1 = max(0.0, min(1.0, x1))
        x2 = max(0.0, min(1.0, x2))
        y1 = max(0.0, min(1.0, y1))
        y2 = max(0.0, min(1.0, y2))

        # OBB format your eval code expects: cls x1 y1 x2 y1 x2 y2 x1 y2
        line_obb = f"{cls} {x1:.6f} {y1:.6f} {x2:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x1:.6f} {y2:.6f}"
        lines_out.append(line_obb)

    out_path = DST_DIR / txt_path.name
    out_path.write_text("\n".join(lines_out), encoding="utf-8")

print("Done converting AABB -> OBB.")


Done converting AABB -> OBB.


## Landing Selection and Evaluation

In [1]:
from pathlib import Path
import numpy as np, cv2, math
from tqdm import tqdm
import os

# --------- CONFIG PATHS (from your description) ----------
DATA_ROOT   = Path("../Data/training_set").resolve()
SPLIT_ROOT  = DATA_ROOT / "slz_out" / "det_obb" / "splits_yolo_obb"
TEST_IMG_DIR= SPLIT_ROOT / "images" / "test"
TEST_LBL_DIR= SPLIT_ROOT / "labels" / "test"
LEVELS_DIR  = DATA_ROOT / "slz_out" / "masks_levels"
PRED_ROOT   = Path("runs/detect_safe/obb_detects").resolve()   # contains subdirs per model

# --------- METRIC CONSTANTS ----------
PURITY_MIN_SLS = 0.90   # SLS@L2 threshold
FSR_HAZARD_FRAC_MIN = 0.03  # 3% hazard-overlap to flag false-safe

def find_test_image(stem: str):
    # prefer .jpg, else any supported ext in TEST_IMG_DIR
    p = TEST_IMG_DIR / f"{stem}.jpg"
    if p.exists(): return p
    return None

def read_levels(stem: str) -> np.ndarray:
    p = LEVELS_DIR / f"{stem}_levels.png"
    lev = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if lev is None:
        raise FileNotFoundError(f"Missing levels mask: {p}")
    if lev.ndim == 3:  # BGR(A) -> R channel as used in your exporter
        lev = lev[:,:,2]
    return lev.astype(np.uint8)

def order_quad_clockwise(pts: np.ndarray) -> np.ndarray:
    # pts: (4,2) float, normalize ordering around centroid
    c = pts.mean(axis=0)
    ang = np.arctan2(pts[:,1]-c[1], pts[:,0]-c[0])
    order = np.argsort(ang)
    return pts[order]

def rasterize_quad_norm(pts_norm: np.ndarray, W: int, H: int) -> np.ndarray:
    # pts_norm in [0,1], shape (4,2)
    pts = pts_norm.copy()
    pts[:,0] = np.clip(pts[:,0] * W, 0, W-1)
    pts[:,1] = np.clip(pts[:,1] * H, 0, H-1)
    pts = order_quad_clockwise(pts).astype(np.float32)
    poly = np.zeros((H, W), np.uint8)
    cv2.fillConvexPoly(poly, pts.astype(np.int32), 1)
    return poly

def parse_pred_file(txt_path: Path):
    """Return list of dicts with keys: cls, conf, pts_norm(4,2)."""
    out = []
    if not txt_path.exists():
        return out
    for line in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s:
            continue
        parts = [float(x) for x in s.split()]
        if len(parts) < 9:
            continue  # not a valid OBB line
        cls = int(parts[0])
        if len(parts) == 9:
            conf = 1.0
            coords = parts[1:]
        else:
            conf = float(parts[1])
            coords = parts[2:]
        if len(coords) != 8:
            continue
        pts = np.array(coords, dtype=np.float32).reshape(4,2)
        out.append({"cls": cls, "conf": conf, "pts_norm": pts})
    return out

def box_area_from_mask(m: np.ndarray) -> int:
    return int(m.sum())

# --------- COLLECT TEST IDS ----------
test_ids = sorted([p.stem for p in TEST_LBL_DIR.glob("*.txt")])
assert test_ids, f"No test label files found in {TEST_LBL_DIR}"

# --------- PRELOAD GT (dims + masks) ----------
gt_cache = {}  # stem -> dict(H,W, levels_resized, safe2_mask, hazard0_mask, safe2_area)
for stem in tqdm(test_ids, desc="Preloading GT masks"):
    img_path = find_test_image(stem)
    if img_path is None:
        raise FileNotFoundError(f"No test image found for {stem} in {TEST_IMG_DIR}")
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f"Cannot read test image {img_path}")
    H, W = img.shape[:2]
    levels = read_levels(stem)
    if levels.shape[:2] != (H, W):
        levels = cv2.resize(levels, (W, H), interpolation=cv2.INTER_NEAREST)
    safe2 = (levels >= 2).astype(np.uint8)
    hazard0 = (levels == 0).astype(np.uint8)
    gt_cache[stem] = dict(H=H, W=W, levels=levels, safe2=safe2, hazard0=hazard0, safe2_area=int(safe2.sum()))

# --------- ENUMERATE MODEL PREDICTION FOLDERS ----------
model_dirs = [d for d in PRED_ROOT.iterdir() if d.is_dir()]
assert model_dirs, f"No model folders found under {PRED_ROOT}"

summaries = []

for mdir in model_dirs:
    model_name = mdir.name
    print(f"\n=== Evaluating model: {model_name} ===")
    # per-model accumulators
    total_images = len(test_ids)
    images_with_preds = 0
    total_boxes = 0

    # mIoA (micro over all boxes)
    purity_sum_micro = 0.0

    # mIoA (macro over images with at least 1 box)
    purity_sum_macro = 0.0
    macro_count_images = 0

    # SAU (micro): numerator/denominator aggregated across images
    sau_safe_cover_sum = 0
    sau_safe_total_sum = 0

    # SLS + FSR
    sls_success = 0                  # denominator: total_images
    fsr_fail = 0                     # denominator: images_with_preds
    selected_count = 0               # images where we selected a box
    selected_purity_sum = 0.0        # for diagnostics
    selected_hazard_frac_sum = 0.0   # for diagnostics

    # iterate images
    for stem in tqdm(test_ids, desc=f"{model_name}", leave=False):
        gt = gt_cache[stem]
        H, W = gt["H"], gt["W"]
        levels = gt["levels"]
        safe2 = gt["safe2"]
        hazard0 = gt["hazard0"]
        safe2_area = gt["safe2_area"]

        # read predictions for this image
        pred_path = mdir / f"{stem}.txt"
        dets = parse_pred_file(pred_path)

        # --- Coverage metrics ---
        if dets:
            images_with_preds += 1

            # union for SAU
            union_mask = np.zeros((H, W), np.uint8)

            # per-image purity average (macro)
            purities_this_image = []

            for det in dets:
                cls = det["cls"]
                if cls not in (0,1):
                    continue  # ignore unexpected classes
                k = 2 if cls == 0 else 3

                P = rasterize_quad_norm(det["pts_norm"], W, H)
                A = box_area_from_mask(P)
                if A <= 0:
                    continue

                # purity p = |P ∩ (S >= k)| / |P|
                allowed = (levels >= k).astype(np.uint8)
                p = float((P & allowed).sum()) / float(A)
                purity_sum_micro += p
                purities_this_image.append(p)

                # union for SAU (always against S>=2)
                union_mask |= P

                total_boxes += 1

            # SAU numerator: |(∪P) ∩ (S>=2)|
            if safe2_area > 0:
                sau_safe_cover_sum += int((union_mask & safe2).sum())
                sau_safe_total_sum += safe2_area

            # per-image purity (macro)
            if purities_this_image:
                purity_sum_macro += float(np.mean(purities_this_image))
                macro_count_images += 1

        else:
            # no predictions: SAU denominator still counts safe area
            if safe2_area > 0:
                sau_safe_total_sum += safe2_area

        # --- Decision metrics (selection) ---
        # rank by Level-3 > Level-2, tie by area; if still tie, higher conf
        selected = None
        best_key = None
        for det in dets:
            cls = det["cls"]
            if cls not in (0,1):
                continue
            P = rasterize_quad_norm(det["pts_norm"], W, H)
            A = box_area_from_mask(P)
            if A <= 0:
                continue
            level_rank = 1 if cls == 1 else 0  # L3=1, L2=0
            key = (level_rank, A, det["conf"])
            if (best_key is None) or (key > best_key):
                best_key = key
                selected = (P, cls, A, det["conf"])

        if selected is None:
            # no selection => SLS failure for this image
            continue

        selected_count += 1
        Psel, cls_sel, A_sel, conf_sel = selected

        # Purity for SLS is vs S>=2 (operationally safe), not class-dependent
        p_sel = float((Psel & safe2).sum()) / float(A_sel)
        selected_purity_sum += p_sel

        # Hazard fraction for FSR
        hfrac_sel = float((Psel & hazard0).sum()) / float(A_sel)
        selected_hazard_frac_sum += hfrac_sel

        if p_sel >= PURITY_MIN_SLS:
            sls_success += 1

        if hfrac_sel >= FSR_HAZARD_FRAC_MIN:
            fsr_fail += 1

    # ---- Aggregate per-model ----
    mIoA_micro = (purity_sum_micro / total_boxes) if total_boxes > 0 else 0.0
    mIoA_macro = (purity_sum_macro / macro_count_images) if macro_count_images > 0 else 0.0
    SAU = (sau_safe_cover_sum / sau_safe_total_sum) if sau_safe_total_sum > 0 else 0.0
    SLS_L2 = sls_success / total_images
    FSR = (fsr_fail / selected_count) if selected_count > 0 else 0.0
    det_rate = images_with_preds / total_images
    avg_sel_purity = (selected_purity_sum / selected_count) if selected_count > 0 else 0.0
    avg_sel_hfrac  = (selected_hazard_frac_sum / selected_count) if selected_count > 0 else 0.0

    summary = dict(
        model=model_name,
        images=total_images,
        imgs_with_preds=images_with_preds,
        det_rate=round(det_rate, 3),
        boxes=total_boxes,
        mIoA_micro=round(mIoA_micro, 4),
        mIoA_macro=round(mIoA_macro, 4),
        SAU=round(SAU, 4),
        SLS_L2=round(SLS_L2, 4),
        FSR=round(FSR, 4),
        avg_sel_purity=round(avg_sel_purity, 4),
        avg_sel_hfrac=round(avg_sel_hfrac, 4),
    )
    summaries.append(summary)

# ---- Print summary table ----
cols = ["model","images","imgs_with_preds","det_rate","boxes",
        "mIoA_micro","mIoA_macro","SAU","SLS_L2","FSR","avg_sel_purity","avg_sel_hfrac"]
hdr = " | ".join([f"{c:>15s}" for c in cols])
print("\n" + hdr)
print("-"*len(hdr))
for s in sorted(summaries, key=lambda d: d["mIoA_micro"], reverse=True):
    print(" | ".join([f"{str(s[c]):>15s}" for c in cols]))


Preloading GT masks: 100%|██████████| 40/40 [00:18<00:00,  2.18it/s]



=== Evaluating model: 5l_dota ===



          model |          images | imgs_with_preds |        det_rate |           boxes |      mIoA_micro |      mIoA_macro |             SAU |          SLS_L2 |             FSR |  avg_sel_purity |   avg_sel_hfrac
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
        5l_dota |              40 |               9 |           0.225 |               9 |           0.001 |           0.001 |             0.0 |             0.0 |          0.6667 |           0.001 |          0.5889
